# Notebook 02 - Precision Levels

This notebook creates versions of the clean base with less numerical precision in distance and timestamps and it measures how much storage each one saves. 

The codec is fixed at zstd (the best one in the last experiment on notebook 01), so the only thing that changes between levels is the precision.

### Step 1 - Imports

In [9]:
import os
import time
import random
import pandas as pd                                   
import pyspark
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt  

In [10]:
# Plot and display settings used in every notebook
def jupyter_settings():
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams.update({
        'figure.figsize': (12, 6),
        'figure.dpi': 100,
        'axes.titlesize': 14,
        'axes.labelsize': 12,
        'legend.fontsize': 10,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'savefig.bbox': 'tight',
        'savefig.dpi': 120
    })
    pd.set_option('display.max_columns', 30)
    pd.set_option('display.float_format', '{:,.2f}'.format)

jupyter_settings()

### Step 2 - Spark Session

Same settings as notebook 01 are same as others : 

* 4 cores
* 3 GB for the driver and the session in UTC.

In [11]:
# Initializing the Spark session
N_PART = 8        # same number of output files for every level
N_REPS = 5        # timed repetitions per read, after one discarded warm-up round (Hoefler and Belli, 2015)
CODEC = "zstd"    # best compression in notebook 01; fixed so only precision changes between levels

spark = (SparkSession.builder
         .master("local[4]")                                   # local mode using the 4 cores of the VM
         .appName("ca1-02-precision-levels")
         .config("spark.driver.memory", "3g")                  # 3 GB: with 4 GB and no swap the VM ran out of memory
         .config("spark.sql.session.timeZone", "UTC")          # avoids the Dublin DST shift on 31/03 with timestamp_ntz
         .config("spark.local.dir", os.path.expanduser("~/spark-tmp"))   # /tmp in this VM is tmpfs (RAM): shuffle files must go to the real disk
         .config("spark.sql.shuffle.partitions", N_PART)       # makes the orderBy below produce N_PART partitions
         .getOrCreate())
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/07 14:12:09 WARN Utils: Your hostname, BDSP, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/07 14:12:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/07 14:12:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/07 14:12:11 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [12]:
# Defining the paths and the HDFS size helper
BASE = "hdfs://localhost:9000/ca1"
CLEAN = f"{BASE}/clean/yellow_2024q1"             # Load clean database from notebook 0 
KEYED = f"{BASE}/clean/yellow_2024q1_keyed"       # clean base plus the experiment keys, written once
LEVELS_OUT = f"{BASE}/levels"                     # storage/ and train/ versions of every level
RESULTS = os.path.expanduser("~/ca1/results")     # absolute path, so it works wherever the notebook is saved
os.makedirs(RESULTS, exist_ok=True)

hadoop = spark._jvm.org.apache.hadoop             # Hadoop Java API, reached through Spark's JVM gateway
fs = hadoop.fs.FileSystem.get(spark._jvm.java.net.URI(BASE), spark._jsc.hadoopConfiguration())

def hdfs_size(path):
    # total bytes of all files under an HDFS path (same value as `hdfs dfs -du -s`)
    return fs.getContentSummary(hadoop.fs.Path(path)).getLength()

26/10/08 00:53:49 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 35489984 ms exceeds timeout 120000 ms
26/10/08 00:53:51 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/08 00:53:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.sto

### Step 3 - Payload Columns

Each level stores five columns: both timestamps, distance and both zones. The true duration (`duration_s_true`) stays at full precision as the reference label, so every level is scored against the same true value.

In [4]:
# Reading the clean base and keeping only the model columns
PAYLOAD = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "PULocationID", "DOLocationID"]   # columns stored at each precision level
base = (spark.read.parquet(CLEAN)
        .select(*PAYLOAD, "duration_s_true")                  
        .orderBy("tpep_pickup_datetime"))                     

### Step 4 - Experiment Keys

Three keys are added before any precision is removed:

- `trip_id`, a unique id for each trip;
- `split`, the temporal split: January and February for training, 1 to 15 March for validation and 16 to 31 March for test;
- `bucket`, a hash of the trip id from 0 to 999, used later to take the same 20% sample at every level.

The base with the keys is written to HDFS once and read back, because ids generated by Spark can change if the DataFrame is recomputed. With them saved, the same trip has the same id, split and bucket at every level.

In [2]:
# Adding the trip id, the temporal split and the sampling bucket
t_val = F.lit("2024-03-01 00:00:00").cast("timestamp_ntz")    # validation starts on 1 March
t_test = F.lit("2024-03-16 00:00:00").cast("timestamp_ntz")   # test starts on 16 March

keyed = (base
         .withColumn("trip_id", F.monotonically_increasing_id())                    # unique id per trip 
         .withColumn("split", F.when(base["tpep_pickup_datetime"] < t_val, "train")
                               .when(base["tpep_pickup_datetime"] < t_test, "validation")
                               .otherwise("test"))                                   # temporal split on the original timestamp
         .withColumn("bucket", F.pmod(F.xxhash64("trip_id"), F.lit(1000))))          # hash of the id 

NameError: name 'F' is not defined

In [9]:
# Writing the keyed base once to HDFS and reading it back, so the ids never change
keyed.write.mode("overwrite").option("compression", CODEC).parquet(KEYED)
keyed = spark.read.parquet(KEYED)
keyed.count()

9213529

The split gives 5,764,483 trips for training, 1,680,517 for validation and 1,768,529 for test.

In [10]:
# Checking the size of each split
keyed.groupBy("split").count().orderBy("split").show()

[Stage 10:==============>                                           (1 + 3) / 4]

+----------+-------+
|     split|  count|
+----------+-------+
|      test|1768529|
|     train|5764483|
|validation|1680517|
+----------+-------+



### Step 5 - Defining the Precision Levels

| Level | Change |
|---|---|
| L0 | native precision (distance with 2 decimals, timestamps to the second) |
| L1 | distance stored as a 32-bit float |
| D1 | distance rounded to 1 decimal |
| D0 | distance rounded to whole miles |
| T1 | both timestamps truncated to the minute |
| T15 | both timestamps truncated to 15 minutes |

`floor_ts` truncates on Unix seconds, which works for any interval; `date_trunc` only works for fixed units like the minute or the hour.

In [12]:
# Defining the timestamp truncation and the precision levels
def floor_ts(col, seconds):
    # floor to a multiple of `seconds` since 1970; works for any interval, unlike date_trunc
    return F.timestamp_seconds(F.floor(F.unix_timestamp(col) / seconds) * seconds).cast("timestamp_ntz")

def apply_level(df, dist_decimals=None, dist_float32=False, ts_seconds=None):
    # returns a copy of df with the requested precision loss applied to the payload columns
    out = df
    if dist_float32:
        out = out.withColumn("trip_distance", out["trip_distance"].cast("float"))            # 4 bytes instead of 8
    if dist_decimals is not None:
        out = out.withColumn("trip_distance", F.round(out["trip_distance"], dist_decimals))
    if ts_seconds is not None:
        out = (out.withColumn("tpep_pickup_datetime", floor_ts(out["tpep_pickup_datetime"], ts_seconds))
                  .withColumn("tpep_dropoff_datetime", floor_ts(out["tpep_dropoff_datetime"], ts_seconds)))
    return out

LEVELS = [
    ("L0_native", {}),
    ("L1_float32", {"dist_float32": True}),
    ("D1_dist_1dec", {"dist_decimals": 1}),
    ("D0_dist_0dec", {"dist_decimals": 0}),
    ("T1_ts_1min", {"ts_seconds": 60}),
    ("T15_ts_15min", {"ts_seconds": 900}),
]

### Step 6 - Checking the Changes on Five Trips

A quick check before writing anything: 11:40:10 becomes 11:30:00 at 15 minutes, and 3.65 miles becomes 4.0 at zero decimals. Distance is rounded, not truncated.

In [13]:
# Checking the truncation and the rounding on five trips
sample = keyed.limit(5)
check = (sample.select("trip_id", "tpep_pickup_datetime", "trip_distance").alias("o")
         .join(apply_level(sample, ts_seconds=900, dist_decimals=0)
               .select("trip_id", "tpep_pickup_datetime", "trip_distance").alias("l"), "trip_id"))   # same trip, before and after the loss
check.select(F.col("o.tpep_pickup_datetime").alias("pickup_original"),
             F.col("l.tpep_pickup_datetime").alias("pickup_15min"),
             F.col("o.trip_distance").alias("dist_original"),
             F.col("l.trip_distance").alias("dist_0dec")).show(truncate=False)

+-------------------+-------------------+-------------+---------+
|pickup_original    |pickup_15min       |dist_original|dist_0dec|
+-------------------+-------------------+-------------+---------+
|2024-03-11 11:40:10|2024-03-11 11:30:00|5.46         |5.0      |
|2024-03-11 11:40:10|2024-03-11 11:30:00|3.65         |4.0      |
|2024-03-11 11:40:11|2024-03-11 11:30:00|2.4          |2.0      |
|2024-03-11 11:40:11|2024-03-11 11:30:00|0.87         |1.0      |
|2024-03-11 11:40:11|2024-03-11 11:30:00|3.81         |4.0      |
+-------------------+-------------------+-------------+---------+



### Step 7 - Warm-up Write

Same as notebook 01: the first write pays the start-up cost and is not timed.

In [14]:
# Writing an untimed warm-up file: the first Spark job pays JVM start-up costs unrelated to the levels
keyed.write.mode("overwrite").parquet(f"{LEVELS_OUT}/_warmup")
fs.delete(hadoop.fs.Path(f"{LEVELS_OUT}/_warmup"), True)   # removed so it does not use HDFS space

True

### Step 8 - Writing the Levels

Each level is written twice. The `storage/` copy has only the five payload columns and is the one measured. The `train/` copy has all the columns and is the one the network reads in notebook 03.

The cell also counts the distinct values of distance and pickup time at each level: exactly for distance, which has only a few thousand values, and approximately (1% error) for the pickup time, which has millions.

In [15]:
# Writing every level in a random order and measuring size, write time and distinct values
order = LEVELS.copy()
random.Random(1234).shuffle(order)   # random order, so no level is always measured first

rows = []
for name, params in order:
    lvl = apply_level(keyed, **params)
    t0 = time.perf_counter()
    lvl.select(*PAYLOAD).write.mode("overwrite").option("compression", CODEC).parquet(f"{LEVELS_OUT}/storage/{name}")   # measured version
    t_write = time.perf_counter() - t0
    lvl.write.mode("overwrite").option("compression", CODEC).parquet(f"{LEVELS_OUT}/train/{name}")                     # version read by the network
    stored = spark.read.parquet(f"{LEVELS_OUT}/storage/{name}")
    card = stored.agg(F.countDistinct("trip_distance").alias("distinct_distance"),                          # exact: only a few thousand values
                      F.approx_count_distinct("tpep_pickup_datetime", 0.01).alias("distinct_pickup")).first()  # approximate, 1% error
    rows.append({"level": name, "payload_mb": hdfs_size(f"{LEVELS_OUT}/storage/{name}") / 1e6, "write_s": t_write,
                 "distinct_distance": card["distinct_distance"], "distinct_pickup": card["distinct_pickup"]})
    print(f"{name:14}{rows[-1]['payload_mb']:9.1f} MB {t_write:7.1f} s  distances: {card['distinct_distance']:>6}  pickups: {card['distinct_pickup']:>9}")

sizes = pd.DataFrame(rows)

26/10/01 13:28:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/10/01 13:28:29 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:28:35 WARN DAGScheduler: Broadcasting large task binary with size 1395.0 KiB
                                                                                

L1_float32         75.9 MB     7.0 s  distances:   3857  pickups:   4678340


26/10/01 13:28:54 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:28:56 WARN DAGScheduler: Broadcasting large task binary with size 1395.0 KiB
                                                                                

D1_dist_1dec       72.4 MB     5.9 s  distances:    493  pickups:   4678340


26/10/01 13:29:13 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:29:17 WARN DAGScheduler: Broadcasting large task binary with size 1395.1 KiB
                                                                                

...

T1_ts_1min         43.1 MB     5.7 s  distances:   3857  pickups:    131437


26/10/01 13:30:17 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:30:21 WARN DAGScheduler: Broadcasting large task binary with size 1395.1 KiB
                                                                                

L0_native          75.9 MB     6.8 s  distances:   3857  pickups:   4678340


26/10/01 13:30:45 WARN DAGScheduler: Broadcasting large task binary with size 1072.8 KiB
26/10/01 13:30:46 WARN DAGScheduler: Broadcasting large task binary with size 1394.9 KiB
                                                                                

D0_dist_0dec       67.7 MB    11.5 s  distances:     67  pickups:   4678340


In [16]:
# Writing every level in a random order and measuring size, write time and distinct values
order = LEVELS.copy()
random.Random(1234).shuffle(order)   # random order, so no level is always measured first

rows = []
for name, params in order:
    lvl = apply_level(keyed, **params)
    t0 = time.perf_counter()
    lvl.select(*PAYLOAD).write.mode("overwrite").option("compression", CODEC).parquet(f"{LEVELS_OUT}/storage/{name}")   # measured version
    t_write = time.perf_counter() - t0
    lvl.write.mode("overwrite").option("compression", CODEC).parquet(f"{LEVELS_OUT}/train/{name}")                     # version read by the network
    stored = spark.read.parquet(f"{LEVELS_OUT}/storage/{name}")
    card = stored.agg(F.countDistinct("trip_distance").alias("distinct_distance"),                          # exact: only a few thousand values
                      F.approx_count_distinct("tpep_pickup_datetime", 0.01).alias("distinct_pickup")).first()  # approximate, 1% error
    rows.append({"level": name, "payload_mb": hdfs_size(f"{LEVELS_OUT}/storage/{name}") / 1e6, "write_s": t_write,
                 "distinct_distance": card["distinct_distance"], "distinct_pickup": card["distinct_pickup"]})
    print(f"{name:14}{rows[-1]['payload_mb']:9.1f} MB {t_write:7.1f} s  distances: {card['distinct_distance']:>6}  pickups: {card['distinct_pickup']:>9}")

sizes = pd.DataFrame(rows)

26/10/01 13:31:04 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:31:06 WARN DAGScheduler: Broadcasting large task binary with size 1395.0 KiB
                                                                                

L1_float32         75.9 MB     5.1 s  distances:   3857  pickups:   4678340


26/10/01 13:31:22 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:31:24 WARN DAGScheduler: Broadcasting large task binary with size 1395.0 KiB
                                                                                

D1_dist_1dec       72.4 MB     5.5 s  distances:    493  pickups:   4678340


26/10/01 13:31:38 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:31:40 WARN DAGScheduler: Broadcasting large task binary with size 1395.1 KiB
                                                                                

...

T1_ts_1min         43.1 MB     3.8 s  distances:   3857  pickups:    131437


26/10/01 13:32:12 WARN DAGScheduler: Broadcasting large task binary with size 1072.9 KiB
26/10/01 13:32:15 WARN DAGScheduler: Broadcasting large task binary with size 1395.1 KiB
                                                                                

L0_native          75.9 MB     5.4 s  distances:   3857  pickups:   4678340


26/10/01 13:32:31 WARN DAGScheduler: Broadcasting large task binary with size 1072.8 KiB
26/10/01 13:32:32 WARN DAGScheduler: Broadcasting large task binary with size 1394.9 KiB


D0_dist_0dec       67.7 MB     6.7 s  distances:     67  pickups:   4678340


### Step 9 - Read Time

Five timed rounds after one discarded round, in a new random order every round, as in notebook 01.

In [17]:
# Measuring the read time in rounds, with a new random order in every round
rng = random.Random(1234)
times = []
for rep in range(N_REPS + 1):
    order = [name for name, _ in LEVELS]
    rng.shuffle(order)                                    # a new random order in every round removes the order bias
    for name in order:
        df = spark.read.parquet(f"{LEVELS_OUT}/storage/{name}")
        t0 = time.perf_counter()
        df.write.format("noop").mode("overwrite").save()  # forces a real read and decode without writing anything
        dt = time.perf_counter() - t0
        if rep > 0:                                       # round 0 is a full warm-up round and is discarded
            times.append({"level": name, "rep": rep, "read_s": dt})
    print("round", rep, "done")

times = pd.DataFrame(times)

round 0 done


round 1 done


round 2 done


round 3 done


round 4 done


[Stage 202:============================>                            (2 + 2) / 4]

round 5 done


### Step 10 - Summary Table

In [19]:
# Building the summary table, in the order the levels were defined
summary = sizes.set_index("level").loc[[name for name, _ in LEVELS]]
summary = summary.join(times.groupby("level")["read_s"].agg(read_median_s="median",
                       read_iqr_s=lambda s: s.quantile(0.75) - s.quantile(0.25)))   # median and interquartile range
summary["size_vs_L0_pct"] = summary["payload_mb"] / summary.loc["L0_native", "payload_mb"] * 100   # storage left after the loss
display(summary.round(2))

,payload_mb,write_s,distinct_distance,distinct_pickup,read_median_s,read_iqr_s,size_vs_L0_pct
level,,,,,,,
L0_native,75.94,5.37,3857,4678340,0.65,0.05,100.00
L1_float32,75.94,5.09,3857,4678340,0.76,0.11,100.00
D1_dist_1dec,72.42,5.50,493,4678340,0.66,0.03,95.36
D0_dist_0dec,67.73,6.74,67,4678340,0.64,0.06,89.18
T1_ts_1min,43.07,3.78,3857,131437,0.57,0.13,56.72
T15_ts_15min,32.21,5.33,3857,8647,0.55,0.12,42.41


The timestamps carry most of the storage. Truncating them to the minute leaves 56.7% of the L0 payload, and to 15 minutes 42.4%. Rounding distance leaves 95.4% (D1) and 89.2% (D0), and float32 changes nothing.

The reason is the number of distinct values. Distance has only 3,857 distinct values in 9.2 million trips, so Parquet already stores it with a dictionary, and a smaller type does not help. Pickup times have about 4.68 million distinct values, which drop to 131,437 at one minute and 8,647 at fifteen. Read times stay between 0.55 and 0.76 s, too small at this size to show a real difference.

In [20]:
# Saving the results
summary.to_csv(f"{RESULTS}/02_levels_summary.csv")
times.to_csv(f"{RESULTS}/02_read_times.csv", index=False)

In [21]:
# Stopping the Spark session and freeing the VM resources
spark.stop()